# Prompt Caching

## Why this matters

This is the one topic in this track with a direct, immediate action item against your own code. Every one of the X++ review agents -- Security, Performance, Best Practice -- sends its full system prompt (the numbered checklist of what to look for) as part of *every single `messages.create()` call*, and none of it uses prompt caching. That system prompt is identical across every review run for a given agent; only the `user_content` (the actual class being reviewed) changes. This notebook covers what caching actually does, how it would apply to `base_agent.py` specifically, and what its real limits are.

## Level 1: What it is

Prompt caching lets you mark a portion of your prompt (typically the system prompt, or any large static content prefix) as cacheable. On the first call, that portion is processed and cached on the provider's infrastructure. On subsequent calls within the cache's lifetime that reuse the *identical* cached content, the provider skips reprocessing it and charges a much lower rate for those tokens -- while still charging full price for whatever varies (the new user message). The API contract, at a glance:

```python
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=8192,
    system=[
        {
            "type": "text",
            "text": SECURITY_SYSTEM_PROMPT,  # the long, static checklist
            "cache_control": {"type": "ephemeral"},
        }
    ],
    messages=[{"role": "user", "content": user_content}],  # varies every call
)
```

The `cache_control` breakpoint tells the API "everything up to and including this block is a candidate for caching."

## Level 2: How it actually works internally

### What's actually being cached

Processing a prompt through a transformer means computing key and value tensors for every token via the attention mechanism, across every layer, before generation of the response can even begin (this is the "prefill" phase, distinct from the token-by-token "decode" phase covered in the quantization/KV-cache notebook). For a long, static system prompt, that prefill computation produces the *exact same* key/value tensors every time, because the input tokens are identical. Prompt caching persists those computed KV tensors (specifically, tied to the exact token sequence up to the cache breakpoint) so a subsequent call with the same prefix can skip straight to using them, rather than recomputing the entire prefill pass for that prefix from scratch.

This is the same underlying KV-cache mechanism from the quantization notebook, extended *across separate API calls* instead of just within a single generation. Within one call, the KV cache avoids recomputing attention for earlier tokens as you generate later ones. Prompt caching avoids recomputing the *entire prefill* for a prefix that's identical to a previous call.

### Cache lifetime and exact-match requirement

Caches are ephemeral, with a lifetime on the order of minutes (typically around 5 minutes, refreshed on each cache hit) -- not persistent storage. And critically: **the cached prefix must match exactly, token-for-token, up to the breakpoint.** Change a single character anywhere in the cached portion -- a typo fix, a reordered sentence, even whitespace -- and it's a cache miss; the entire prefix gets reprocessed and re-cached as if new. This has a direct implication for `base_agent.py`'s design: the `ISSUE_TOOL` schema and each agent's `SYSTEM_PROMPT` need to be genuinely static (not dynamically re-built with any per-call variation) for caching to actually help -- which they already are, since they're defined once at module level, not constructed inside `run()`.

### Where the breakpoint should go

Cache breakpoints are placed *after* the content you want cached, and everything before that breakpoint (in order) becomes the cached prefix. For the X++ agents, the ideal split is: `SYSTEM_PROMPT` (the numbered checklist, `ISSUE_SCHEMA` reference, entirely static per agent) goes in the cached `system` block; the `user_content` (which embeds `xpp.raw_code` -- different for every file reviewed) stays uncached, since it's different every call by definition and caching it would never hit anyway.

## Level 3: Where it breaks, and what it doesn't solve

1. **It doesn't help at all if nothing repeats.** If every call's full input (system + user content) is unique, there's nothing to cache -- you'd write the cache once and never hit it again before it expires. This matters for judging whether it's worth doing in the X++ project specifically: the system prompts repeat across every file reviewed with that agent (real savings), but `user_content` -- the actual class source -- essentially never repeats between different review runs (no benefit there, and none is expected).

2. **First call is not cheaper -- it's actually a premium, not neutral.** Writing to the cache costs 25% *more* than a normal input token would (the platform is doing extra work to store the prefix), while reading from an existing cache costs roughly 90% *less* than standard input pricing. So a single call with no reuse afterward is a net loss, not a wash -- the economics only work out once you get at least a couple of cache hits within the window. For a long system prompt hit even 2-3 times in a session, the savings add up fast; hit once and never again, and you paid a premium for nothing.

3. **The ~5-minute ephemeral window (refreshed on each hit) means caching helps burst/session traffic more than sporadic use.** An extended 1-hour TTL option also exists (`cache_control: {"type": "ephemeral", "ttl": "1h"}`) at a higher write-cost premium, for cases where the gap between reuses is predictably longer than a few minutes but reuse still happens same-day. If you run the X++ reviewer once, close the laptop, and come back tomorrow with the default TTL, there's no cache left to hit -- the benefit is real specifically when you're running several reviews back-to-back (e.g., reviewing multiple files, or running the review-then-adjust-then-rerun cycle done repeatedly across this project's sessions), not for isolated single runs days apart.

4. **Exact-match fragility is a real engineering constraint, not a minor detail.** If you're mid-iteration on a system prompt (like the many times the X++ project's prompts got trimmed after moving checks to deterministic code), every edit invalidates the cache for that prompt going forward until the next stable version accumulates enough repeat calls to be worth it again. This is a real reason not to over-optimize prompt wording prematurely if you're actively still iterating on it.

5. **It doesn't reduce output token cost at all**, only input token cost on the cached portion. If a bottleneck is verbose `suggested_fix` fields in the response (output tokens), caching the system prompt does nothing for that -- a different lever (asking for more concise output, or restructuring the schema) would be needed instead.

In [ ]:
# What base_agent.py's messages.create() call would look like with
# caching added -- this is the actual, concrete change, not illustrative
# pseudocode. Compare against the current call_claude() implementation.

# CURRENT (no caching -- system_prompt reprocessed in full every call):
#
# response = client.messages.create(
#     model=MODEL,
#     max_tokens=MAX_TOKENS,
#     temperature=0,
#     system=system_prompt,
#     tools=[ISSUE_TOOL],
#     tool_choice={"type": "tool", "name": "report_issues"},
#     messages=[{"role": "user", "content": user_content}],
# )

# WITH CACHING -- system prompt marked as a cacheable block:
def call_claude_with_caching(client, system_prompt: str, user_content: str,
                               model: str, max_tokens: int, issue_tool: dict):
    return client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=0,
        system=[
            {
                "type": "text",
                "text": system_prompt,
                "cache_control": {"type": "ephemeral"},
            }
        ],
        tools=[issue_tool],
        tool_choice={"type": "tool", "name": "report_issues"},
        messages=[{"role": "user", "content": user_content}],
    )

# Note: ISSUE_TOOL itself is also static across all calls to a given
# agent, and tool definitions can be cached too via the same mechanism
# on the tools block -- worth checking current Anthropic docs for the
# exact supported placement, since caching support surface has been
# actively expanding.
print("Would apply this pattern to base_agent.py's call_claude() -- "
      "one line added per system prompt, no other logic changes needed.")

## Interview-ready summary

**Q: "How would you reduce cost on a system making repeated LLM calls with a long, mostly-static prompt?"**

Weak answer: "I'd shorten the prompt."

Strong answer: "First I'd check whether the static portion is actually being cached -- if the system prompt is identical across calls (which it is for something like a code review agent's specialist instructions), prompt caching lets the provider skip reprocessing that prefix's prefill computation on repeat calls within the cache window, at a much lower token rate. It only helps if there's genuine repetition within roughly a 5-minute window and the cached content is byte-for-byte identical -- any edit invalidates it -- so it's not a fix for content that's still being actively iterated on, and it doesn't touch output token cost at all. I'd only shorten the prompt itself as a separate lever if the actual bottleneck were response latency from a very long prefill on the *first* uncached call, not recurring cost, since caching already addresses the recurring cost."